# Internet Usage Analysis — World Bank Data
**Based on Keith Galli's Charity Livestream**  
Data source: [World Bank Development Indicators](https://databank.worldbank.org/source/world-development-indicators)

---
### Questions we'll answer:
1. How has global internet usage grown over time?
2. Which countries lead / lag in internet adoption?
3. How do income groups compare?
4. Which countries grew the fastest in the last decade?
5. Is there a correlation between GDP per capita and internet usage?
6. Where does Indonesia rank?

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Setup done ✓')

## 1. Load the Data

The World Bank CSV comes in **wide format**: one row per country, year columns (1960–2023).  
We'll load it, inspect it, then melt it to **long format** for analysis.

In [ ]:
# ------------------------------------------------------------------
# Load raw data
# ------------------------------------------------------------------
# File from: https://github.com/KeithGalli/internet-usage-analysis/tree/master/data
# Indicator: IT.NET.USER.ZS  (Individuals using the Internet, % of population)

df_raw = pd.read_csv('data/internet_usage.csv', skiprows=4)
print(df_raw.shape)
df_raw.head(3)

In [ ]:
# Column names
df_raw.columns.tolist()

## 2. Clean & Reshape

In [ ]:
# ------------------------------------------------------------------
# Drop useless columns + rename
# ------------------------------------------------------------------
KEEP_ID = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']

# Year columns are strings like '1960', '1961', ... '2023'
year_cols = [c for c in df_raw.columns if c.isdigit()]

df = df_raw[KEEP_ID + year_cols].copy()

print(f'Countries/regions: {df.shape[0]}')
print(f'Year range: {year_cols[0]} – {year_cols[-1]}')
print(f'Missing values: {df[year_cols].isna().sum().sum():,}')

In [ ]:
# ------------------------------------------------------------------
# Wide → Long format
# ------------------------------------------------------------------
df_long = df.melt(
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Internet Usage (%)'
)

df_long['Year'] = df_long['Year'].astype(int)
df_long = df_long.dropna(subset=['Internet Usage (%)'])
df_long = df_long.sort_values(['Country Name', 'Year']).reset_index(drop=True)

print(df_long.shape)
df_long.head(10)

In [ ]:
# ------------------------------------------------------------------
# Separate actual COUNTRIES from regional aggregates
# World Bank includes aggregates like 'World', 'High income', etc.
# They have codes that don't match ISO-3 country codes.
# ------------------------------------------------------------------
AGGREGATES = [
    'ARB', 'CSS', 'CEB', 'EAR', 'EAS', 'EAP', 'TEA', 'EMU',
    'ECS', 'ECA', 'TEC', 'EUU', 'FCS', 'HPC', 'HIC', 'IBD',
    'IBT', 'IDB', 'IDX', 'IDA', 'LTE', 'LCN', 'LAC', 'TLA',
    'LDC', 'LMY', 'LIC', 'LMC', 'MEA', 'MNA', 'TMN', 'MIC',
    'NAC', 'INX', 'OED', 'OSS', 'PSS', 'PST', 'PRE', 'SST',
    'SAS', 'TSA', 'SSF', 'SSA', 'TSS', 'UMC', 'WLD'
]

df_countries = df_long[~df_long['Country Code'].isin(AGGREGATES)].copy()
df_agg = df_long[df_long['Country Code'].isin(AGGREGATES)].copy()

print(f'Country rows:   {len(df_countries):,}')
print(f'Aggregate rows: {len(df_agg):,}')

## 3. Question 1 — How has global internet usage grown over time?

In [ ]:
# Pull the 'World' aggregate
world = df_agg[df_agg['Country Code'] == 'WLD'].copy()

fig, ax = plt.subplots()
ax.plot(world['Year'], world['Internet Usage (%)'], linewidth=2.5, color='steelblue')
ax.fill_between(world['Year'], world['Internet Usage (%)'], alpha=0.15, color='steelblue')

ax.set_title('Global Internet Adoption (% of population)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('% of Population')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlim(world['Year'].min(), world['Year'].max())

# Annotate latest value
latest = world.iloc[-1]
ax.annotate(f"{latest['Internet Usage (%)']:.1f}%",
            xy=(latest['Year'], latest['Internet Usage (%)']),
            xytext=(-40, 10), textcoords='offset points',
            fontsize=11, color='steelblue', fontweight='bold')

plt.tight_layout()
plt.savefig('q1_global_growth.png', dpi=150)
plt.show()

## 4. Question 2 — Which countries lead / lag in internet adoption? (latest year)

In [ ]:
# Get the most recent year with data for each country
latest_year = df_countries.groupby('Country Name')['Year'].max().reset_index()
latest_year.columns = ['Country Name', 'Latest Year']

df_latest = df_countries.merge(latest_year, on='Country Name')
df_latest = df_latest[df_latest['Year'] == df_latest['Latest Year']]

print(f'Countries with data: {len(df_latest)}')
print(f'Year range: {df_latest["Latest Year"].min()} – {df_latest["Latest Year"].max()}')

In [ ]:
# Top 20
top20 = df_latest.nlargest(20, 'Internet Usage (%)')
bot20 = df_latest.nsmallest(20, 'Internet Usage (%)')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Top 20 ---
axes[0].barh(top20['Country Name'], top20['Internet Usage (%)'], color='seagreen')
axes[0].set_title('Top 20 — Highest Internet Usage', fontweight='bold')
axes[0].set_xlabel('% of Population')
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].invert_yaxis()
for bar, val in zip(axes[0].patches, top20['Internet Usage (%)']):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8)

# --- Bottom 20 ---
axes[1].barh(bot20['Country Name'], bot20['Internet Usage (%)'], color='tomato')
axes[1].set_title('Bottom 20 — Lowest Internet Usage', fontweight='bold')
axes[1].set_xlabel('% of Population')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].invert_yaxis()
for bar, val in zip(axes[1].patches, bot20['Internet Usage (%)']):
    axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('q2_top_bottom_countries.png', dpi=150)
plt.show()

## 5. Question 3 — How do income groups compare over time?

In [ ]:
# Income group codes from World Bank aggregates
INCOME_MAP = {
    'HIC': 'High income',
    'UMC': 'Upper-middle income',
    'LMC': 'Lower-middle income',
    'LIC': 'Low income'
}

df_income = df_agg[df_agg['Country Code'].isin(INCOME_MAP)].copy()
df_income['Income Group'] = df_income['Country Code'].map(INCOME_MAP)

fig, ax = plt.subplots()

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
for (group, grp_df), color in zip(df_income.groupby('Income Group'), colors):
    ax.plot(grp_df['Year'], grp_df['Internet Usage (%)'],
            label=group, linewidth=2, color=color)

ax.set_title('Internet Usage by Income Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('% of Population')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('q3_income_groups.png', dpi=150)
plt.show()

In [ ]:
# Gap analysis: high vs low income
pivot_income = df_income.pivot_table(
    index='Year', columns='Income Group', values='Internet Usage (%)'
)
pivot_income['Gap (High-Low)'] = pivot_income['High income'] - pivot_income['Low income']

print('Digital divide (High income - Low income, % points):')
print(pivot_income[['High income', 'Low income', 'Gap (High-Low)']].tail(10).to_string())

## 6. Question 4 — Fastest-growing countries in the last 10 years

In [ ]:
# Get usage value for 10-year window: (latest year) vs (latest year - 10)
RECENT_YEAR = df_countries['Year'].max()
BASE_YEAR = RECENT_YEAR - 10

recent = df_countries[df_countries['Year'] == RECENT_YEAR][['Country Name', 'Country Code', 'Internet Usage (%)']].copy()
base   = df_countries[df_countries['Year'] == BASE_YEAR  ][['Country Name', 'Country Code', 'Internet Usage (%)']].copy()

recent.columns = ['Country Name', 'Country Code', 'Recent']
base.columns   = ['Country Name', 'Country Code', 'Base']

growth = recent.merge(base, on=['Country Name', 'Country Code'])
growth['Growth (pp)'] = growth['Recent'] - growth['Base']  # percentage points
growth['Growth (%)']  = ((growth['Recent'] - growth['Base']) / growth['Base'] * 100).round(1)

growth = growth.dropna().sort_values('Growth (pp)', ascending=False)
print(f'Countries with data for both {BASE_YEAR} and {RECENT_YEAR}: {len(growth)}')
growth.head(10)

In [ ]:
top_growers = growth.head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_growers['Country Name'], top_growers['Growth (pp)'], color='mediumpurple')
ax.set_title(f'Fastest-Growing Countries: {BASE_YEAR}→{RECENT_YEAR} (percentage point gain)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Percentage Point Increase')
ax.invert_yaxis()

for bar, val in zip(bars, top_growers['Growth (pp)']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'+{val:.1f} pp', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('q4_fastest_growing.png', dpi=150)
plt.show()

## 7. Question 5 — Correlation between internet usage and GDP per capita

We need a second dataset: GDP per capita (NY.GDP.PCAP.CD)  
Download the same way from the World Bank portal and save as `data/gdp_per_capita.csv`.

In [ ]:
try:
    df_gdp_raw = pd.read_csv('data/gdp_per_capita.csv', skiprows=4)
    
    gdp_year_cols = [c for c in df_gdp_raw.columns if c.isdigit()]
    df_gdp = df_gdp_raw[['Country Name', 'Country Code'] + gdp_year_cols].copy()
    
    df_gdp_long = df_gdp.melt(
        id_vars=['Country Name', 'Country Code'],
        value_vars=gdp_year_cols,
        var_name='Year',
        value_name='GDP per Capita (USD)'
    )
    df_gdp_long['Year'] = df_gdp_long['Year'].astype(int)
    df_gdp_long = df_gdp_long.dropna()
    gdp_available = True
    print('GDP data loaded ✓')

except FileNotFoundError:
    gdp_available = False
    print('GDP data not found — skipping correlation chart.')
    print('Download from: https://databank.worldbank.org → Indicator: NY.GDP.PCAP.CD')

In [ ]:
if gdp_available:
    # Join internet + GDP for the same year
    internet_snap = df_countries[df_countries['Year'] == RECENT_YEAR][['Country Name', 'Country Code', 'Internet Usage (%)']]
    gdp_snap = df_gdp_long[df_gdp_long['Year'] == RECENT_YEAR][['Country Name', 'Country Code', 'GDP per Capita (USD)']]

    combined = internet_snap.merge(gdp_snap, on=['Country Name', 'Country Code'])
    print(f'Countries in both datasets: {len(combined)}')

    # Log-scale scatter
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.scatter(combined['GDP per Capita (USD)'], combined['Internet Usage (%)'],
               alpha=0.6, edgecolors='white', linewidth=0.5, s=60, color='steelblue')

    # Highlight a few notable countries
    highlights = ['United States', 'China', 'India', 'Indonesia', 'Nigeria', 'Germany', 'Japan']
    for _, row in combined[combined['Country Name'].isin(highlights)].iterrows():
        ax.annotate(row['Country Name'],
                    xy=(row['GDP per Capita (USD)'], row['Internet Usage (%)']),
                    fontsize=8, color='black',
                    xytext=(5, 3), textcoords='offset points')

    # Correlation
    corr = combined['GDP per Capita (USD)'].corr(combined['Internet Usage (%)'])
    ax.text(0.05, 0.92, f'Pearson r = {corr:.3f}', transform=ax.transAxes,
            fontsize=11, color='darkred', fontweight='bold')

    ax.set_xscale('log')
    ax.set_xlabel('GDP per Capita, USD (log scale)')
    ax.set_ylabel('Internet Usage (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'GDP per Capita vs Internet Usage ({RECENT_YEAR})', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.savefig('q5_gdp_vs_internet.png', dpi=150)
    plt.show()
    
    print(f'\nLog-GDP Pearson correlation: {combined["GDP per Capita (USD)"].apply(np.log).corr(combined["Internet Usage (%)"]):.3f}')

## 8. Question 6 — Where does Indonesia rank?

In [ ]:
# Indonesia timeseries
indo = df_countries[df_countries['Country Code'] == 'IDN'].copy()
print(indo[['Year', 'Internet Usage (%)']].tail(10).to_string(index=False))

In [ ]:
# Global rank for latest year
rank_df = df_latest.sort_values('Internet Usage (%)', ascending=False).reset_index(drop=True)
rank_df['Rank'] = rank_df.index + 1

indo_rank = rank_df[rank_df['Country Code'] == 'IDN']
print(indo_rank[['Rank', 'Country Name', 'Internet Usage (%)', 'Latest Year']].to_string(index=False))
print(f'\nTotal countries ranked: {len(rank_df)}')

In [ ]:
# Compare Indonesia vs regional neighbors
SEA_COUNTRIES = ['Indonesia', 'Philippines', 'Vietnam', 'Thailand',
                 'Malaysia', 'Singapore', 'Myanmar', 'Cambodia']

sea = df_countries[df_countries['Country Name'].isin(SEA_COUNTRIES)]

fig, ax = plt.subplots()
for country, grp in sea.groupby('Country Name'):
    style = {'linewidth': 3} if country == 'Indonesia' else {'linewidth': 1.5, 'linestyle': '--', 'alpha': 0.7}
    ax.plot(grp['Year'], grp['Internet Usage (%)'], label=country, **style)

ax.set_title('Internet Usage — Southeast Asia', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('% of Population')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('q6_southeast_asia.png', dpi=150)
plt.show()

## 9. Bonus — Heatmap: Top 30 countries, last 20 years

In [ ]:
# Top 30 countries by latest usage
top30_names = df_latest.nlargest(30, 'Internet Usage (%)')['Country Name'].tolist()

heatmap_df = df_countries[
    (df_countries['Country Name'].isin(top30_names)) &
    (df_countries['Year'] >= RECENT_YEAR - 20)
].pivot_table(
    index='Country Name',
    columns='Year',
    values='Internet Usage (%)'
)

# Sort by most recent year
heatmap_df = heatmap_df.sort_values(heatmap_df.columns[-1], ascending=False)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(heatmap_df, annot=False, cmap='YlGnBu', fmt='.0f',
            linewidths=0.3, ax=ax, cbar_kws={'label': '% of Population'})

ax.set_title('Internet Usage Heatmap — Top 30 Countries (last 20 years)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('q7_heatmap.png', dpi=150)
plt.show()

## 10. Summary Statistics

In [ ]:
summary = df_latest.describe()
print(summary)

print(f"\n{'='*50}")
print(f"Global internet users today (World):")
world_latest = df_agg[(df_agg['Country Code'] == 'WLD') & (df_agg['Year'] == RECENT_YEAR)]
if not world_latest.empty:
    pct = world_latest['Internet Usage (%)'].values[0]
    print(f"  {pct:.1f}% of world population = ~{pct/100*8_000_000_000/1e9:.1f} billion people")

print(f"\nCountries below 20%: {(df_latest['Internet Usage (%)'] < 20).sum()}")
print(f"Countries above 90%: {(df_latest['Internet Usage (%)'] > 90).sum()}")

---
## Key Takeaways

| Finding | Detail |
|---|---|
| Global adoption | ~60–66% of world population online as of latest year |
| Digital divide | ~60+ pp gap between high and low income countries |
| Fastest growth | Mostly Sub-Saharan Africa and South Asia in last decade |
| GDP correlation | Strong positive correlation (log-GDP r ≈ 0.85) |
| Indonesia | Mid-range globally; growing fast in SEA |

---
*Data: World Bank World Development Indicators — Indicator: IT.NET.USER.ZS*